In [153]:
%load_ext autoreload
%autoreload 2


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [154]:
# base imports
import numpy as np 
import pandas as pd
from functools import partial
import pickle
import warnings
import os

# preprocessing
from nltk.stem import WordNetLemmatizer
from nltk.stem import PorterStemmer
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

from bs4 import BeautifulSoup

import re

# NLP packages
from sklearn.feature_extraction.text import TfidfVectorizer,CountVectorizer
from sklearn.model_selection import train_test_split
import scipy

from datasets import load_dataset

max_length = 32
size = 'all'
"""
Data Preprocessing Functions
"""
stemmer = PorterStemmer()
lemma = WordNetLemmatizer()

def tokenize(text,vocabulary,max_length = 32):
    # my text was unicode so I had to use the unicode-specific translate function. If your documents are strings, you will need to use a different `translate` function here. `Translated` here just does search-replace. See the trans_table: any matching character in the set is replaced with `None`
    # tokens = [word for word in word_tokenize(text.lower()) if len(word) > 1] #if len(word) > 1 because I only want to retain words that are at least two characters before stemming, although I can't think of any such words that are not also stopwords
    tokens = [word for word in text.lower() if len(word) > 1] #if len(word) > 1 because I only want to retain words that are at least two characters before stemming, although I can't think of any such words that are not also stopwords
    # tokens = [word for word in word_tokenize(text.lower())]# if len(word) > 1] #if len(word) > 1 because I only want to retain words that are at least two characters before stemming, although I can't think of any such words that are not also stopwords
    indexed_tokens = [vocabulary.get(token, -1)+2 for token in tokens]  # 1 for unknown words

    # Trim to 32 tokens or pad with -1s if shorter
    if len(indexed_tokens) < max_length:
        indexed_tokens.extend([0] * (max_length - len(indexed_tokens)))  # Padding with -1
    else:
        indexed_tokens = indexed_tokens[:max_length]  # Trim to n_tokens=max_length

    
    # return [vocabulary.get(token, -1) for token in tokens]  # -1 for unknown words
    return indexed_tokens

def token2index(tokens,vocabulary,max_length = 32):
    # my text was unicode so I had to use the unicode-specific translate function. If your documents are strings, you will need to use a different `translate` function here. `Translated` here just does search-replace. See the trans_table: any matching character in the set is replaced with `None`
    # tokens = [word for word in word_tokenize(text.lower()) if len(word) > 1] #if len(word) > 1 because I only want to retain words that are at least two characters before stemming, although I can't think of any such words that are not also stopwords
    
    # tokens = [word for word in word_tokenize(text.lower())]# if len(word) > 1] #if len(word) > 1 because I only want to retain words that are at least two characters before stemming, although I can't think of any such words that are not also stopwords
    indexed_tokens = [vocabulary.get(token, -1)+2 for token in tokens]  # 1 for unknown words

    # Trim to 32 tokens or pad with -1s if shorter
    if len(indexed_tokens) < max_length:
        indexed_tokens.extend([0] * (max_length - len(indexed_tokens)))  # Padding with -1
    else:
        indexed_tokens = indexed_tokens[:max_length]  # Trim to n_tokens=max_length

    
    # return [vocabulary.get(token, -1) for token in tokens]  # -1 for unknown words
    return indexed_tokens

def assert_even_length_array(arr):
    if len(arr) % 2 != 0:
        raise AssertionError("Array does not have an even number of elements: {}".format(len(arr)))

def check_order(array):
    prev_entry = None
    ilist = []  
    for i, entry in enumerate(array):
        if entry == prev_entry:
            ilist.append(i)
    
        prev_entry = entry

    return ilist

def sort_data_cfs(array,ilist):
    # given a list of indices, swap the index and the following index?
    for i in ilist:
        temp1 = array[i]
        temp2 = array[i+1]

        array.iloc[i] = temp2
        array.iloc[i+1] = temp1

    return array

def sort_label_cfs(array,ilist):
    # given a list of indices, swap the index and the following index?
    for i in ilist:
        temp1 = array[i]
        temp2 = array[i+1]

        array[i] = temp2
        array[i+1] = temp1

    return array


def get_vector(vec1,vec2):
    
    d_vec = vec2 - vec1
    
    if np.sum(d_vec) == 0:
        
        warnings.warn("A text vector and its' counterfactual vector have the same value. This is probably not right.")
        
    if isinstance(d_vec, scipy.sparse.csr_matrix):  # or any other sparse matrix type
        mag = scipy.linalg.norm(d_vec.todense())

    elif isinstance(d_vec, np.ndarray):
        mag = np.linalg.norm(d_vec)
        # Perform operations specific to dense arrays
    else:
        print("Unknown vector type")
        mag = np.linalg.norm(d_vec)
    
    if mag!=0:
        return d_vec/mag        
    else:
        return np.zeros((np.shape(vec1)[0]))
    # return [d_vec/mag]



## cleaning the text

def cleantext(text):
    # removing the "\"
    text = re.sub("'\''","",text)
    
    # removing special symbols
    text = re.sub("[^ a-zA-Z]","",text)
    
    # removing the whitespaces
    text = ' '.join(text.split())
    
    # convert text to lowercase    
    text = text.lower()
    
    return text

# removing the stopwords
stop_words = set(stopwords.words('english'))
mod_stop_words = set(stopwords.words('english')) - {'not', 'but'}
# stop_words = stop_words - set(['dont','do'])
# def removestopwords(text):
    
#     removedstopword = [word for word in text.split() if word not in mod_stop_words]
#     return ' '.join(removedstopword)

def removestopwords(text):
    
    removedstopword = [word for word in text if word not in mod_stop_words]
    return removedstopword

#Removing the html strips 
def strip_html(text):
    soup = BeautifulSoup(text, "html.parser")
    return soup.get_text()

#Removing the square brackets
def remove_between_square_brackets(text):
    return re.sub(r'\[[^]]*\]', '', text)

#Removing Emails
def remove_Emails(text):
    pattern=r'\S+@\S+'
    text=re.sub(pattern,'',text)
    return text

#Removing URLS
def remove_URLS(text):
    pattern=r'http\S+'
    text=re.sub(pattern,'',text)
    return text

def remove_special_characters(text, remove_digits=True):
    pattern=r'[^a-zA-z0-9\s]'
    text=re.sub(pattern,' ',text)
    return text

#Removing numbers
def remove_numbers(text):
    pattern = r'\d+'
    text = re.sub(pattern, '', text)
    return text

def lematizing(sentence):
    stemSentence = ""
    for word in sentence.split():
        stem = lemma.lemmatize(word)
        stemSentence += stem
        stemSentence += " "
    stemSentence = stemSentence.strip()
    return stemSentence

def stemming(sentence):
    
    stemmed_sentence = ""
    for word in sentence.split():
        stem = stemmer.stem(word)
        stemmed_sentence+=stem
        stemmed_sentence+=" "
        
    stemmed_sentence = stemmed_sentence.strip()
    return stemmed_sentence



def process_data(data):
    # Step 1: clean text (string level)
    data = data.apply(lambda x: strip_html(x))
    data = data.apply(remove_between_square_brackets)
    data = data.apply(lambda x: cleantext(x))
    data = data.apply(remove_special_characters)
    data = data.apply(remove_URLS)
    data = data.apply(remove_Emails)
    data = data.apply(remove_numbers)
    
    # Step 2: tokenize
    data = data.apply(lambda x: word_tokenize(x.lower()))
    
    # Step 3: remove stopwords and stem (token level)
    data = data.apply(lambda tokens: [stemming(token) for token in removestopwords(tokens)])

    return data




def gen_knowledge(data,label,cf=False):
    """
    Takes sequences of pairs of counterfactuals and 
    returns just the originals, with the counterfactual 
    included as an annotation.

    If cf=True, returns the counterfactuals and omits the originals
    """
    
    vectors_in = data['vector']
    text_in = data['text']
    # sparse_array = sp.csr_matrix((0,dlength))

    text_out=[]
    vectors_out = []
    labels = []
    # vectors = np.empty((0,1,dlength))
    cf_text = []
    cf_labels = []
    cf_vectors = []
    
    for i in range(int(np.shape(vectors_in)[0]/2)):
        
        
        if cf:
            vec1,vec2 = vectors_in[i * 2 + 1],vectors_in[i * 2]  # Get the data from the current dataset slice
            label_out = label[i*2+1]
            text = text_in[i * 2 + 1]
            _cf_text=text_in[i * 2]
        else:
            vec1,vec2 = vectors_in[i * 2],vectors_in[i * 2 + 1]  # Get the data from the current dataset slice                
            label_out = label[i*2]
            text = text_in[i * 2]
            _cf_text = text_in[i * 2 + 1]
            
        text_out.append(text)
        labels.append(label_out)
        vectors_out.append(vec1)
    
        cf_vectors.append([vec2])
        cf_labels.append([1 - label_out])
        cf_text.append([_cf_text])
    
    return {'vector':vectors_out,'text':text_out},labels,cf_vectors, cf_labels,cf_text



def compile_K(data,label, paired=False,cf=False,int_out=False):
    
    if paired:
        
        if np.shape(data['vector'])[0]%2 != 0:
            raise ValueError("Can't generate paired counterfactuals with an uneven number of samples")
        
        data,label,vector,labels,cf_text = gen_knowledge(data,label,cf=cf)
        
        # labels = [1 - l for l in label]
        
    else:
        warnings.warn("Non-paired data just creates blanks atm")
        
        vector = [[] for _ in range(np.shape(data['vector'])[0])]
        cf_text = [[] for _ in range(np.shape(data['vector'])[0])]

        labels = [[]]*len(vector)
    
    # labels = np.expand_dims(labels, axis=1)

    magnitude = np.ones(len(vector))
    magnitude = np.expand_dims(magnitude, axis=1)
    

    if int_out:
        for i in range(np.shape(vector)[0]):
            vector[i] = vector[i].astype(int)

    n_samples = len(data['text'])
    print(f'Returning {n_samples} samples with {len(vector)} counterfactuals')
    
    return {'text':data['text'],
            'X':data['vector'],
            'Y':label,
            'K':{'vector':vector,'label':labels,'magnitude':magnitude, 'text':cf_text}}


def compile_ag(X,y,text,cf_X,cf_text):
    
    magnitude = np.ones(len(cf_X))
    magnitude = np.expand_dims(magnitude, axis=1)

    n_samples = len(text)

    print(f'Returning {n_samples} samples with {len(cf_text)} counterfactuals')
    
    return {'text':list(text),
            'X':X,
            'Y':list(y),
            'K':{'vector':np.expand_dims(cf_X,axis=1),
                 'label':np.expand_dims(np.full_like(np.array(y),np.nan),axis=1),
                 'magnitude':magnitude, 
                 'text':np.expand_dims(cf_text,axis=1)}}

In [155]:
ds = load_dataset("SetFit/ag_news")

agnews = pd.read_json('data/original/fizle_AG_News_Llama3-70B-Instruct.json')
agnews_all = pd.read_csv('data/original/ag_news.csv')
agnews['label'] = agnews_all['label'][:500]

# X_train, X_test, y_train, y_test = train_test_split(agnews['input'], agnews['label'], test_size=0.2, random_state=42)
# X_test, X_dev, y_test, y_dev = train_test_split(X_test, y_test, test_size=0.5, random_state=42)
if size == 'all':
    X_train, y_train = agnews['input'], agnews['label']
else:
    X_train, y_train = agnews['input'][:size], agnews['label'][:size]
X_test, X_dev, y_test, y_dev = train_test_split(agnews_all['text'], agnews_all['label'], test_size=0.5, random_state=42)

In [156]:
X_train_processed = process_data(X_train)
X_dev_processed = process_data(X_dev)
X_test_processed = process_data(X_test)

In [157]:
cf_train = agnews['counterfactual'][list(X_train.index)]
# cf_dev = agnews['counterfactual'][list(X_dev.index)]
# cf_test = agnews['counterfactual'][list(X_test.index)]


In [158]:

data_for_vectorizer = X_train_processed.apply(lambda tokens: " ".join(tokens))
vectorizer = CountVectorizer(max_features=20000)
vectorizer.fit(data_for_vectorizer)
# Get the vocabulary mapping (word to integer index)
vocabulary = vectorizer.vocabulary_

# define a reusable lambda
to_indices = lambda texts: [token2index(text, vocabulary, max_length=max_length) for text in texts]

# now you can call it on any dataset
tfidf_train = to_indices(X_train_processed)
tfidf_dev   = to_indices(X_dev_processed)
tfidf_test  = to_indices(X_test_processed)

tfidf_train_cf = to_indices(process_data(cf_train))
# tfidf_dev_cf = to_indices(process_data(cf_dev))
# tfidf_test_cf = to_indices(process_data(cf_test))

In [159]:


"""
########################################################################################################################
Save embeddings
########################################################################################################################
"""
# Stripping html from unprocessed text, just to clean it up
print('\ntrain_Data')
cf_train={'original': compile_ag(tfidf_train,y_train,X_train,tfidf_train_cf,cf_train)} # X,y,text,cf_X,cf_text

for key,val in cf_train['original']['K'].items():
    print(key,np.shape(val))
    print(val[0:10])

print('\ndev_Data')
cf_dev={'original':{ 
                    'text':X_dev,
                    'X':tfidf_dev,
                    'Y':y_dev,
                    'K':{'vector':None,'label':None,'magnitude':None, 'text':None}
                    }}

print('\ntest_Data')
cf_test={'original':{ 
                    'text':X_test,
                    'X':tfidf_test,
                    'Y':y_test,
                    'K':{'vector':None,'label':None,'magnitude':None, 'text':None}
                    }}
        


pickle_data = {'train':cf_train,'test':cf_test,'dev':cf_dev}

embedding_path = f'data/integer_len{max_length}_SIZE_{size}.pkl'
print(f"Saving to {embedding_path}")
with open(embedding_path, 'wb') as file:
    pickle.dump(pickle_data, file)



train_Data
Returning 500 samples with 500 counterfactuals
vector (500, 1, 32)
[[[1222 3633 2495 3422    1 2839 2258 3652 2258 2976  947 3422 3332 2451
   2258 1225 2840    0    0    0    0    0    0    0    0    0    0    0
      0    0    0    0]]

 [[2738 3018 2647 3444 3061 1922  841 1639   75 1291 3228 3228 3550  501
   3018 3444   75 3859  678 2152  153    1 2648  722 2647 1374  949 2820
   2354  148 1262 1922]]

 [[1896  675 3826 1469  923 2500  168  168  675 1336    1 3653 2014 1469
    923 2124 2074  344 2500 3100  552  124   21  468  370 2680    0    0
      0    0    0    0]]

 [[2617 3652 1570 1319 3822  168  168  291  844 2143 1272 3633  949 2820
   1171 3283 3087  377  656 2064 1243 1096  568  479  109 1884  845  443
   1969 3334    1 2806]]

 [[1347   76 1972 1205 3174  168  168 3219 1347 3175   67 3796 1085  406
   3693 1362   42 2258 1262 2930 2796   77 2576  827  771 2062    0    0
      0    0    0    0]]

 [[2378 1959  446 2237 1699 1698 2991  446  903 3567 1699    

/Users/jonathanerskine/University of Bristol/gradient_supervision/counterfactual-gradient-alignment/cfa_venv/lib/python3.10/site-packages/numpy/core/numeric.py:407: RuntimeWarning: invalid value encountered in cast
  multiarray.copyto(res, fill_value, casting='unsafe')


creating baseline with counterfactuals

In [160]:
cf_train['original']['text'][0:2]

["Fears for T N pension after talks Unions representing workers at Turner Newall say they are 'disappointed' after talks with stricken parent firm Federal Mogul.",
 'The Race is On: Second Private Team Sets Launch Date for Human Spaceflight (SPACE.com) SPACE.com - TORONTO, Canada -- A second\\team of rocketeers competing for the  #36;10 million Ansari X Prize, a contest for\\privately funded suborbital space flight, has officially announced the first\\launch date for its manned rocket.']

In [161]:
print(cf_train['original']['X'][0:2])
print(cf_train['original']['Y'][0:2])


[[1222, 1, 2495, 3422, 3650, 2839, 3859, 3610, 2290, 2976, 947, 3422, 3332, 2451, 1261, 1225, 2177, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [2738, 3018, 2647, 3444, 3061, 1922, 841, 1638, 3230, 3228, 3228, 3550, 501, 3022, 2904, 678, 2152, 153, 1, 2648, 722, 1330, 1374, 3353, 3227, 1291, 2354, 148, 1266, 841, 2053, 2904]]
[2, 3]


In [162]:
print(cf_train['original']['K']['text'][0:2])
print(cf_train['original']['K']['vector'][0:2])
print(cf_train['original']['K']['magnitude'][0:2])
print(cf_train['original']['K']['label'][0:2])

[["Fears for UN pension after talks Diplomats representing nations at United Nations say they are 'disappointed' after talks with stricken parent nation Federal Republic."]
 ['The Race is On: Second Private Team Sets Launch Date for Humanitarian Aid Flight (SPACE.com) SPACE.com - TORONTO, Canada -- A second team of aid workers competing for the $10 million Ansari X Prize, a contest for privately funded disaster relief, has officially announced the first launch date for its manned cargo plane.']]
[[[1222 3633 2495 3422    1 2839 2258 3652 2258 2976  947 3422 3332 2451
   2258 1225 2840    0    0    0    0    0    0    0    0    0    0    0
      0    0    0    0]]

 [[2738 3018 2647 3444 3061 1922  841 1639   75 1291 3228 3228 3550  501
   3018 3444   75 3859  678 2152  153    1 2648  722 2647 1374  949 2820
   2354  148 1262 1922]]]
[[1.]
 [1.]]
[[0]
 [0]]
